# 01. Exploratory Data Analysis

**Đồ án**: Trip Delay Prediction

**Mục tiêu notebook này**:
1. Load 14 bảng dữ liệu
2. Khám phá schema và relationships
3. Phân tích target (delay_hours)
4. Univariate / Bivariate analysis
5. Temporal patterns
6. Insights ban đầu

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import load_all_tables, get_table_info
from src.utils import set_seed

set_seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_DIR = Path('../data_raw')
FIG_DIR = Path('../results/figures/eda')
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load 14 bảng

In [ ]:
tables = load_all_tables(DATA_DIR)
info = get_table_info(tables)
info

## 2. Schema từng bảng

In [ ]:
for name, df in tables.items():
    print(f'\n=== {name.upper()} ===')
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(df.head(2))

## 3. Tính target (delay_hours) từ trips và delivery_events

**Định nghĩa**: `delay_hours = actual_arrival_time - scheduled_arrival_time`

In [ ]:
# Target = delay tại event_type='Delivery' (theo CSV thực tế dùng cột scheduled_datetime, actual_datetime)
events = tables['delivery_events']
delivery_events = events[events['event_type'] == 'Delivery'].copy()

delivery_events['scheduled_datetime'] = pd.to_datetime(delivery_events['scheduled_datetime'])
delivery_events['actual_datetime'] = pd.to_datetime(delivery_events['actual_datetime'])
delivery_events['delay_hours'] = (
    delivery_events['actual_datetime'] - delivery_events['scheduled_datetime']
).dt.total_seconds() / 3600

print(f'Total delivery events: {len(delivery_events)}')
print(delivery_events['delay_hours'].describe())

## 4. Phân tích target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(delivery_events['delay_hours'].dropna(), bins=80, edgecolor='black')
axes[0].set_title('Distribution of delay_hours')
axes[0].set_xlabel('Delay (hours)')
axes[0].axvline(0, color='red', linestyle='--', label='On-time')
axes[0].legend()

axes[1].boxplot(delivery_events['delay_hours'].dropna(), vert=False)
axes[1].set_title('Boxplot of delay_hours')
axes[1].set_xlabel('Delay (hours)')

plt.tight_layout()
plt.savefig(FIG_DIR / 'target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Missing values

In [ ]:
for name, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f'\n--- {name} ---')
        print((nulls / len(df) * 100).round(2).astype(str) + '%')

## 6. Temporal patterns

Phân tích delay theo:
- Tháng / Quý
- Day of week
- Hour of day

In [ ]:
delivery_events['month'] = delivery_events['scheduled_datetime'].dt.month
delivery_events['day_of_week'] = delivery_events['scheduled_datetime'].dt.day_name()
delivery_events['hour'] = delivery_events['scheduled_datetime'].dt.hour

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
delivery_events.groupby('month')['delay_hours'].mean().plot(kind='bar', ax=axes[0])
axes[0].set_title('Avg delay by month')

delivery_events.groupby('day_of_week')['delay_hours'].mean().plot(kind='bar', ax=axes[1])
axes[1].set_title('Avg delay by day_of_week')

delivery_events.groupby('hour')['delay_hours'].mean().plot(ax=axes[2])
axes[2].set_title('Avg delay by hour')

plt.tight_layout()
plt.savefig(FIG_DIR / 'temporal_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Insights & Notes

Các nhận xét chính sau EDA:

1. **Target distribution**: ...
2. **Missing values**: ...
3. **Temporal patterns**: ...
4. **Outliers**: ...
5. **Class imbalance** (nếu nhị phân hóa): ...

→ Sang notebook 02 để preprocessing.